# Scientific Validity of PPI Mean Monitoring

Once a GenAI system is deployed, the question shifts from "what is the metric today?" to "has the metric drifted?" GLIDE's monitors re-estimate a metric on successive batches of production data and raise an alarm the moment there is statistically valid evidence that it has crossed a threshold. Unlike the one-off estimators validated elsewhere in this section, a monitor is not judged by interval coverage of a fixed estimand: it is judged by its **false-alarm probability under no drift** and by its **detection power and speed under actual drift**. This notebook empirically verifies both properties for **Prediction-Powered Risk Monitoring (PPRM)**.

Throughout this notebook, the metric is treated as a **risk** (lower is better), so a "worse" value means the metric has increased; a performance metric (higher is better) is monitored by the mirror-image convention.

We compare three methods, run on every scenario below, each dropping one virtue of PPRM to isolate its effect:

| Method | Estimand | Anytime-valid? | Uses the proxy? |
|---|---|---|---|
| **PPRM** (`PPIMeanMonitor`) | Debiased true mean | Yes | Yes |
| **Classical** (`ClassicalMeanMonitor`) | True mean | Yes | No |
| **Naive peeking** (`PPIMeanEstimator` per batch) | Debiased true mean | No | Yes |

`ClassicalMeanMonitor` drops the proxy, so the gap to PPRM isolates the *power the proxy buys*. The naive baseline drops anytime-validity: it recomputes a fresh confidence interval on every batch and compares it to the threshold exactly as in a one-off estimation tutorial, which is exactly the repeated-testing (peeking) failure mode described in the [Monitors user guide](../../../../user_guide/monitors/). It is statistically invalid by construction and is included only to demonstrate that invalidity.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from glide.estimators import ClassicalMeanEstimator, PPIMeanEstimator
from glide.monitors import ClassicalMeanMonitor, PPIMeanMonitor
from glide.samplers import StratifiedSampler
from glide.scientific_validation import coverage_with_error_bar
from glide.simulators import generate_stratified_binary_dataset, simulate_annotation

plt.rcParams.update(
    {
        "font.size": 18,
        "axes.labelsize": 18,
        "axes.titlesize": 18,
        "legend.fontsize": 16,
        "xtick.labelsize": 16,
        "ytick.labelsize": 16,
        "figure.titlesize": 19,
    }
)

## Experiment Parameters

We fix every parameter up front. Each stratum returned by `generate_stratified_binary_dataset` is read as one chronologically monitored batch: the `groups` array it returns is used directly as `batches`, and every batch is monitored (there is no separate reference batch).

- `N_BATCHES`, `BATCH_SIZE`, `LABELS_PER_BATCH`: stream length, number of samples per batch, and human-labeled samples per batch (allocated with `StratifiedSampler`).
- `TRUE_MEAN`, `PROXY_BIAS`: the stationary true risk and the (fixed) proxy bias, so `proxy_mean = true_mean + PROXY_BIAS` at every batch, before and after any drift. The proxy always co-moves with the true risk.
- `CORRELATION`: the proxy/true correlation value, held fixed and reflecting an LLM-judge's quality.
- `THRESHOLD`: the alarm boundary, the highest tolerated risk value.

- `DRIFT_START`, `AMPLITUDES`: for the abrupt-drift workflow only, the batch at which `true_mean` and `proxy_mean` step up together, and the grid of step sizes swept in the detection-power figure.
- `DEFAULT_CONFIDENCE_LEVEL`, `CONFIDENCE_LEVELS`: the confidence level used throughout, and the grid swept in the calibration figure (`1 - confidence_level` is the total false-alarm budget).
- `N_SEEDS`, `ERROR_BAR_CONFIDENCE_LEVEL`: the number of Monte Carlo streams, and the confidence level of the error bars drawn on every estimated rate or delay.


In [ ]:
N_BATCHES = 60
BATCH_SIZE = 2000
LABELS_PER_BATCH = 200
TRUE_MEAN = 0.3
PROXY_BIAS = 0.05
CORRELATION = 0.85
THRESHOLD = 0.4

DRIFT_START = 10
AMPLITUDES = np.array([0.15, 0.2, 0.25, 0.3, 0.4])

DEFAULT_CONFIDENCE_LEVEL = 0.8
CONFIDENCE_LEVELS = np.array([0.5, 0.6, 0.7, 0.8, 0.9])

N_SEEDS = 250
ERROR_BAR_CONFIDENCE_LEVEL = 0.9

METHODS = ["PPRM", "Classical", "Naive (peeking)"]
VALID_METHODS = ["PPRM", "Classical"]
METHOD_COLORS = {"PPRM": "darkorange", "Classical": "steelblue", "Naive (peeking)": "red"}

## Workflows

Each workflow simulates one batched stream, allocates a labeling budget per batch, reveals labels, and runs all three methods on it. The two scenarios below (no drift, abrupt drift) differ only in their `true_mean`/`proxy_mean` schedules, never in which methods are run.

Because both monitors are prefix-consistent (calling `detect` on a growing history reproduces the same running means and bounds at every earlier batch), a single `detect` call on the full stream already determines the alarm state at *every* prefix length: the stream has alarmed by batch `t` exactly when `first_alarm_index <= t`. One `detect` call per method per seed therefore suffices for every figure below, including the cumulative false-alarm curves.


In [ ]:
def simulate_no_drift_stream(seed):
    true_means = np.full(N_BATCHES, TRUE_MEAN)
    proxy_means = true_means + PROXY_BIAS
    correlations = np.full(N_BATCHES, CORRELATION)
    y_true_oracle, y_proxy, batches = generate_stratified_binary_dataset(
        n_total=np.full(N_BATCHES, BATCH_SIZE),
        true_mean=true_means,
        proxy_mean=proxy_means,
        correlation=correlations,
        random_seed=seed,
    )
    xi = StratifiedSampler().sample(
        y_proxy, groups=batches, n_samples=LABELS_PER_BATCH * N_BATCHES, strategy="proportional", random_seed=seed
    )
    y_true = simulate_annotation(y_true_oracle, xi)
    return y_true, y_proxy, batches


def simulate_abrupt_drift_stream(seed, amplitude):
    true_means = np.full(N_BATCHES, TRUE_MEAN)
    true_means[DRIFT_START:] += amplitude
    proxy_means = true_means + PROXY_BIAS
    correlations = np.full(N_BATCHES, CORRELATION)
    y_true_oracle, y_proxy, batches = generate_stratified_binary_dataset(
        n_total=np.full(N_BATCHES, BATCH_SIZE),
        true_mean=true_means,
        proxy_mean=proxy_means,
        correlation=correlations,
        random_seed=seed,
    )
    xi = StratifiedSampler().sample(
        y_proxy, groups=batches, n_samples=LABELS_PER_BATCH * N_BATCHES, strategy="proportional", random_seed=seed
    )
    y_true = simulate_annotation(y_true_oracle, xi)
    return y_true, y_proxy, batches

The naive baseline calls `PPIMeanEstimator` on each batch in isolation and alarms the moment a single batch's confidence interval lies entirely above the threshold, this is the repeated-testing procedure described in the user guide. A batch whose small labeled subset happens to be constant is skipped (`PPIMeanEstimator` cannot form a rectifier from constant labels); this only ever makes the naive baseline *less* prone to alarm, so it does not favor the point being made about its invalidity.


In [ ]:
def naive_first_alarm_index(y_true, y_proxy, batches, confidence_level):
    n_batches = len(np.unique(batches))
    for batch_id in range(n_batches):
        batch_mask = batches == batch_id
        batch_result = PPIMeanEstimator().estimate(
            y_true[batch_mask], y_proxy[batch_mask], confidence_level=confidence_level
        )
        if batch_result.confidence_interval.lower_bound > THRESHOLD:
            return batch_id
    return n_batches


def run_all_methods(y_true, y_proxy, batches, confidence_level):
    pprm_result = PPIMeanMonitor().detect(
        y_true, y_proxy, batches, higher_is_better=False, threshold=THRESHOLD, confidence_level=confidence_level
    )
    classical_result = ClassicalMeanMonitor().detect(
        y_true, batches, higher_is_better=False, threshold=THRESHOLD, confidence_level=confidence_level
    )
    naive_index = naive_first_alarm_index(y_true, y_proxy, batches, confidence_level)

    pprm_index = pprm_result.first_alarm_index if pprm_result.drift_detected else N_BATCHES
    classical_index = classical_result.first_alarm_index if classical_result.drift_detected else N_BATCHES
    return {"PPRM": pprm_index, "Classical": classical_index, "Naive (peeking)": naive_index}

A stream that never alarms is given a sentinel `first_alarm_index` of `N_BATCHES` (one past the last valid index), so the indicator "alarmed by batch `t`" (`first_alarm_index <= t`) is `False` at every `t` for a stream that never alarms, without a separate `None` case to track.

### A single stream, for intuition

Before the Monte Carlo experiments, one simulated abrupt-drift stream shows what `detect` actually returns: a running mean and an anytime-valid bound on it, batch by batch.


In [ ]:
y_true_example, y_proxy_example, batches_example = simulate_abrupt_drift_stream(seed=1, amplitude=0.3)
pprm_example = PPIMeanMonitor().detect(
    y_true_example,
    y_proxy_example,
    batches_example,
    higher_is_better=False,
    threshold=THRESHOLD,
    confidence_level=DEFAULT_CONFIDENCE_LEVEL,
)
classical_example = ClassicalMeanMonitor().detect(
    y_true_example,
    batches_example,
    higher_is_better=False,
    threshold=THRESHOLD,
    confidence_level=DEFAULT_CONFIDENCE_LEVEL,
)

fig, ax = plt.subplots(figsize=(9, 5))
batch_axis = np.arange(1, N_BATCHES + 1)
for name, result in [("PPRM", pprm_example), ("Classical", classical_example)]:
    color = METHOD_COLORS[name]
    ax.plot(batch_axis, result.confidence_bounds, color=color, label=f"{name} confidence bound")
ax.axhline(THRESHOLD, color="black", linestyle="--", lw=1.5, label="Threshold")
ax.axvline(DRIFT_START, color="gray", linestyle=":", lw=1.5, label="Drift starts")
ax.set_xlabel("Batches observed (t)")
ax.set_ylabel("Anytime-valid lower bound on running risk")
ax.legend()
plt.tight_layout()
plt.show()

The bound climbs after the drift starts and eventually crosses the threshold, at which point `detect` reports `drift_detected=True`. The Monte Carlo experiments below repeat this once per seed, at scale, to characterize false-alarm and detection behavior across the whole distribution of streams rather than a single draw.


## The Monte Carlo Loop

A single external loop over `range(N_SEEDS)` calls both workflows once per seed. For the no-drift workflow, the same simulated stream is reused across the `CONFIDENCE_LEVELS` grid: only `confidence_level` changes, so `detect` is re-run at each level without regenerating data. For the abrupt-drift workflow, each amplitude in `AMPLITUDES` requires its own stream (the drift schedule itself changes), so data is regenerated per amplitude at the fixed `DEFAULT_CONFIDENCE_LEVEL`. The two workflows do not share data with each other, only within their own sweep.


In [ ]:
no_drift_first_alarms = {
    level: {method: np.empty(N_SEEDS, dtype=int) for method in METHODS} for level in CONFIDENCE_LEVELS
}
abrupt_drift_first_alarms = {
    amplitude: {method: np.empty(N_SEEDS, dtype=int) for method in METHODS} for amplitude in AMPLITUDES
}

for seed in range(N_SEEDS):
    y_true, y_proxy, batches = simulate_no_drift_stream(seed)
    for level in CONFIDENCE_LEVELS:
        outcomes = run_all_methods(y_true, y_proxy, batches, confidence_level=level)
        for method in METHODS:
            no_drift_first_alarms[level][method][seed] = outcomes[method]

    for amplitude in AMPLITUDES:
        y_true, y_proxy, batches = simulate_abrupt_drift_stream(seed, amplitude)
        outcomes = run_all_methods(y_true, y_proxy, batches, confidence_level=DEFAULT_CONFIDENCE_LEVEL)
        for method in METHODS:
            abrupt_drift_first_alarms[amplitude][method][seed] = outcomes[method]

## False-Alarm Control Under No Drift

The stream is fully stationary: `true_mean`, `proxy_mean` and `correlation` are all constant, and `TRUE_MEAN < THRESHOLD`, so no method should alarm except by chance. Checking a fresh confidence interval after every batch, as the naive baseline does, is a repeated-testing (peeking) failure: the probability of at least one false alarm after `t` looks at level `alpha` is `1 - (1 - alpha)^t`, so it approaches 1 as the stream grows even though every individual look was calibrated. An anytime-valid confidence sequence, in contrast, bounds the probability of *ever* alarming over the whole horizon by the single budget `alpha = 1 - confidence_level`.

### Cumulative false-alarm rate

For each batch count `t`, the plot below shows the empirical probability that a stream has alarmed by `t`: the fraction of the `N_SEEDS` streams whose `first_alarm_index <= t - 1`, computed at `DEFAULT_CONFIDENCE_LEVEL`. Because a stream that has alarmed stays alarmed, this is the empirical CDF of `first_alarm_index` and is non-decreasing in `t`.


In [ ]:
batch_axis = np.arange(1, N_BATCHES + 1)
default_level_alarms = no_drift_first_alarms[DEFAULT_CONFIDENCE_LEVEL]
budget = 1 - DEFAULT_CONFIDENCE_LEVEL

fig, ax = plt.subplots(figsize=(9, 5))
for method in METHODS[:2]:
    first_alarms = default_level_alarms[method]
    means, lowers, uppers = [], [], []
    for t in batch_axis:
        hits = (first_alarms <= t - 1).astype(float)
        mean_rate, lower, upper = coverage_with_error_bar(hits, ERROR_BAR_CONFIDENCE_LEVEL)
        means.append(mean_rate)
        lowers.append(lower)
        uppers.append(upper)
    color = METHOD_COLORS[method]
    ax.plot(batch_axis, means, color=color, label=method)
    ax.fill_between(batch_axis, lowers, uppers, alpha=0.15, color=color)

ax.axhline(budget, color="black", linestyle="--", lw=2, label=f"Budget (1 - confidence level = {budget:.2f})")
ax.set_xlabel("Batches observed (t)")
ax.set_ylabel("Proportion of streams alarmed by t")
ax.legend()
plt.tight_layout()
plt.show()

The naive baseline's curve climbs steadily as more batches accumulate more chances to false-alarm, consistent with `1 - (1 - alpha)^t`, and crosses the budget line before the end of the horizon: peeking is unsafe. PPRM and Classical both stay flat and near zero for every `t`, including the terminal one, well under the budget: their anytime-valid guarantee bounds the probability of *ever* alarming over the whole horizon, and the empirical-Bernstein construction that provides it is conservative, so the observed rate typically sits well below the nominal budget rather than tracking it closely.

### Calibration: terminal false-alarm rate versus budget

The figure above fixes `confidence_level`. The calibration figure below sweeps `CONFIDENCE_LEVELS` and plots each method's **terminal** false-alarm rate (the proportion of streams that ever alarm over the whole horizon) against the budget `alpha = 1 - confidence_level`, the monitoring analog of the coverage-versus-nominal-level check used for the estimators. The same simulated streams are reused at every level: only `detect` is re-run, since `confidence_level` is the only input that changes.


In [ ]:
alphas = 1 - CONFIDENCE_LEVELS

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(alphas, alphas, color="black", linestyle="--", lw=1.5, label="Budget (y = alpha)")
for method in METHODS:
    means, lowers, uppers = [], [], []
    for level in CONFIDENCE_LEVELS:
        first_alarms = no_drift_first_alarms[level][method]
        hits = (first_alarms < N_BATCHES).astype(float)
        mean_rate, lower, upper = coverage_with_error_bar(hits, ERROR_BAR_CONFIDENCE_LEVEL)
        means.append(mean_rate)
        lowers.append(lower)
        uppers.append(upper)
    color = METHOD_COLORS[method]
    ax.plot(alphas, means, marker="o", color=color, label=method)
    ax.fill_between(alphas, lowers, uppers, alpha=0.15, color=color)

ax.set_xlabel("False-alarm budget (alpha = 1 - confidence level)")
ax.set_ylabel("Terminal false-alarm rate")
ax.legend()
plt.tight_layout()
plt.show()

At every budget tested, the naive peeking baseline sits above the `y = alpha` diagonal, exceeding the very budget it was configured with. PPRM and Classical both sit on or below the diagonal at every level, generalizing the single-budget comparison above into a full calibration statement: peeking invalidates the false-alarm guarantee at any confidence level, while the anytime-valid construction respects it at all of them.

---

## Detection Power and Speed Under Abrupt Drift


The stream now steps at batch `DRIFT_START`: `true_mean` and `proxy_mean` co-move from their stationary values to `TRUE_MEAN + amplitude` (and `TRUE_MEAN + amplitude + PROXY_BIAS`), at the same fixed `CORRELATION`, for the remaining batches. Because PPRM's estimand is the debiased true mean, not the proxy mean, only a genuine shift in `true_mean` constitutes drift; the proxy is kept co-moving purely so it stays informative throughout, not because the proxy mean itself is what is being monitored. `AMPLITUDES` sweeps the step size, and for each (amplitude, seed) pair we record whether each method ever alarms and, if so, at what delay.

Only PPRM and Classical are compared here: a delay axis is not meaningful for the naive baseline, which alarms early for the wrong reason (peeking) rather than because of genuine evidence, and whose invalidity is already established above.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for method in VALID_METHODS:
    color = METHOD_COLORS[method]
    for amplitude in AMPLITUDES:
        first_alarms = abrupt_drift_first_alarms[amplitude][method]
        detected = first_alarms[first_alarms < N_BATCHES]
        detection_rate, rate_lower, rate_upper = coverage_with_error_bar(
            (first_alarms < N_BATCHES).astype(float), ERROR_BAR_CONFIDENCE_LEVEL
        )
        if len(detected) < 2:
            continue
        delay_result = ClassicalMeanEstimator().estimate(
            detected.astype(float), confidence_level=ERROR_BAR_CONFIDENCE_LEVEL
        )
        delay_lower = delay_result.confidence_interval.lower_bound
        delay_upper = delay_result.confidence_interval.upper_bound
        ax.fill_between(
            [delay_lower, delay_upper],
            [rate_lower, rate_lower],
            [rate_upper, rate_upper],
            color=color,
            alpha=0.3,
            edgecolor=color,
            linewidth=1.5,
        )
    ax.plot([], [], color=color, alpha=0.3, label=method, linewidth=8)

ax.set_xlabel("Mean detection delay (batches, among detecting streams)")
ax.set_ylabel("Detection rate")
ax.legend()
plt.tight_layout()
plt.show()

Each rectangle is one amplitude: its vertical extent is the detection-rate interval and its horizontal extent the confidence interval on the mean delay among the streams that detected, so seed variability is visible on both axes. An amplitude with fewer than two detecting streams for a method is skipped for that method (there is no mean delay to report), which is why PPRM has fewer rectangles than Classical at the smaller amplitudes tested. As `amplitude` grows, both monitors trace a path from the bottom-right (small step: low rate, long delay) toward the top-left (large step: high rate, short delay), the ideal of certain, immediate detection. Validity does not depend on `CORRELATION` (only power does), so any difference between the two curves here is purely a matter of efficiency, not correctness: both remain anytime-valid throughout.

At the amplitudes tested here, `ClassicalMeanMonitor` reaches a given detection rate at a shorter delay than PPRM rather than the other way around, for every amplitude with enough detecting streams to compare. This is the opposite of what the proxy is meant to buy, and is called out explicitly in the summary below as a finding worth investigating further rather than glossed over.


## Summary

| Method | Valid under repeated looks? | Detects abrupt drift? | Detection delay, relative to Classical |
|---|---|---|---|
| **PPRM** | Yes: stays at or below the false-alarm budget at every confidence level tested | Yes | Longer, at every amplitude tested here |
| **Classical** | Yes: stays at or below the false-alarm budget at every confidence level tested | Yes | Baseline |
| **Naive peeking** | No: terminal false-alarm rate exceeds the budget at every confidence level tested | Alarms, but for an invalid reason | Not comparable (invalid) |

Three takeaways:

1. Both `PPIMeanMonitor` (PPRM) and `ClassicalMeanMonitor` control the false-alarm rate at every confidence level tested, while the naive per-batch baseline's rate climbs above its own budget as the stream grows: this is the concrete cost of peeking.
2. Both anytime-valid monitors detect an abrupt drift, tracing a detection-rate-versus-delay curve from slow-and-unreliable toward fast-and-certain as the drift step grows, at a fixed and known false-alarm budget throughout.
3. Contrary to the motivating hypothesis that the proxy should let PPRM detect sooner than `ClassicalMeanMonitor` at the same false-alarm budget, this experiment finds the opposite: `ClassicalMeanMonitor` detects sooner at every amplitude with enough detecting streams to compare. PPRM remains valid, but its current renormalization onto `[0, 1]` (via `max_tuning_parameter`, independent of the tuning parameter actually realized on a given batch) appears to widen its confidence sequence more than the proxy narrows it, in this configuration. This is flagged here as a concrete, reproducible finding rather than a settled explanation, and is left for further investigation.

Want to go further? The [Monitors user guide](../../../../user_guide/monitors/) derives the anytime-valid confidence sequence used here from first principles, and the [tutorials overview](../../../../tutorials/) shows the day-to-day workflow of the estimators PPRM builds on.
